# Data Processing Pipeline for Parallel Aligned Translations

This notebook processes multilingual parallel translation data from TMX (Translation Memory eXchange) files. It:

1. **Loads** aligned translation segments across 4 languages (English, Spanish, French, German) from TMX files
2. **Cleans** the data by applying quality filters:
    - Minimum word count threshold
    - Sentences must start with a letter
    - Excludes all-caps text
    - Limits punctuation and number ratio
3. **Exports** the cleaned parallel data in two formats:
    - A single TSV file with all languages aligned by row
    - Per-language text files maintaining line alignment
4. **Reports** summary statistics on retained segments after processing

The pipeline ensures full alignment across all languages, meaning only segments available in all 4 languages are kept.

In [1]:
import os
import re
import glob
import xml.etree.ElementTree as ET
from collections import defaultdict
from typing import List, Tuple, Dict, Iterable, Optional

import pandas as pd
import numpy as np

In [ ]:
DATA_DIR = "" # set to the path of the downloaded data
OUTPUT_DIR = "output" # set to the desired output directory
LANGS = ("EN-GB", "ES-ES", "FR-FR", "DE-DE")   # fixed column order for alignment
MIN_WORDS = 5
MAX_NUM_PUNCT_RATIO = 0.35
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
_word_re = re.compile(r"\S+")
_letter_start_re = re.compile(r"^[A-Za-zÀ-ÿ]", re.UNICODE)

def is_all_caps(s: str) -> bool:
    letters = re.sub(r"[^A-Za-zÀ-ÿ]", "", s, flags=re.UNICODE)
    return len(letters) > 0 and letters.isupper()

def ratio_numbers_punct(s: str) -> float:
    if not s:
        return 1.0
    nonletters = re.findall(r"[\d\W_]", s, flags=re.UNICODE)
    return len(nonletters) / len(s)

def starts_with_letter(s: str) -> bool:
    return bool(_letter_start_re.match(s))

def has_min_words(s: str, n: int) -> bool:
    return len(_word_re.findall(s)) >= n

In [ ]:
def parse_tmx_file(filepath: str, langs: Tuple[str, ...]) -> List[Tuple[Optional[str], ...]]:
    """
    Return list of rows; each row is a tuple of len(langs) containing the segment text
    for each language (same order as `langs`). If any requested language is missing in a TU,
    that TU is skipped to preserve full alignment.
    """
    rows: List[Tuple[Optional[str], ...]] = []
    xml_lang_attr = "{http://www.w3.org/XML/1998/namespace}lang"

    context = ET.iterparse(filepath, events=("start", "end"))
    _, root = next(context)  

    current_tu: Dict[str, str] = {}
    in_tu = False

    for event, elem in context:
        tag = elem.tag.split("}")[-1] 

        if event == "start" and tag == "tu":
            in_tu = True
            current_tu = {}

        elif event == "end":
            if tag == "tuv" and in_tu:
                lang_code = elem.attrib.get(xml_lang_attr) or elem.attrib.get("lang")
                if lang_code:
                    seg_el = elem.find(".//seg")
                    text = (seg_el.text or "").strip() if seg_el is not None else ""
                    current_tu[lang_code] = text

            elif tag == "tu" and in_tu:
                if all(l in current_tu for l in langs):
                    row = tuple(current_tu[l] for l in langs)
                    rows.append(row)

                elem.clear()
                root.clear()
                in_tu = False

    return rows

In [ ]:
def parse_tmx_file(filepath: str, langs: Tuple[str, ...]) -> List[Tuple[Optional[str], ...]]:
    """
    Return list of rows; each row is a tuple of len(langs) containing the segment text
    for each language (same order as `langs`). If any requested language is missing in a TU,
    that TU is skipped to preserve full alignment.
    """
    # Streaming parse to handle large files
    rows: List[Tuple[Optional[str], ...]] = []
    xml_lang_attr = "{http://www.w3.org/XML/1998/namespace}lang"

    context = ET.iterparse(filepath, events=("start", "end"))
    _, root = next(context)  # get root

    current_tu: Dict[str, str] = {}
    in_tu = False

    for event, elem in context:
        tag = elem.tag.split("}")[-1]  # strip namespace if present

        if event == "start" and tag == "tu":
            in_tu = True
            current_tu = {}

        elif event == "end":
            if tag == "tuv" and in_tu:
                lang_code = elem.attrib.get(xml_lang_attr) or elem.attrib.get("lang")
                if lang_code:
                    seg_el = elem.find(".//seg")
                    text = (seg_el.text or "").strip() if seg_el is not None else ""
                    current_tu[lang_code] = text

            elif tag == "tu" and in_tu:
                # finalize TU: keep only if all requested langs are present
                if all(l in current_tu for l in langs):
                    row = tuple(current_tu[l] for l in langs)
                    rows.append(row)

                # clear to free memory and move on
                elem.clear()
                root.clear()
                in_tu = False

    return rows

In [6]:
def load_aligned_dataframe(data_dir: str, langs: Tuple[str, ...]) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(data_dir, "*.tmx")))
    print(f"Found {len(files)} TMX files.")
    aligned_rows: List[Tuple[str, ...]] = []

    for i, fpath in enumerate(files, 1):
        rows = parse_tmx_file(fpath, langs)
        print(f"[{i}/{len(files)}] {os.path.basename(fpath)} → kept {len(rows)} fully-aligned TUs")
        aligned_rows.extend(rows)

    df = pd.DataFrame(aligned_rows, columns=list(langs))
    print(f"\nLoaded {len(df)} fully-aligned rows across {len(files)} files.")
    return df

In [ ]:
def clean_aligned_dataframe(df: pd.DataFrame,
                            langs: Tuple[str, ...],
                            min_words: int = MIN_WORDS,
                            max_num_punct_ratio: float = MAX_NUM_PUNCT_RATIO) -> pd.DataFrame:
    before = len(df)
    print(f"\nStarting cleaning on {before} aligned rows...")

    def mask_min_words(col: pd.Series) -> pd.Series:
        return col.apply(lambda s: has_min_words(s, min_words))

    def mask_starts_with_letter(col: pd.Series) -> pd.Series:
        return col.apply(starts_with_letter)

    def mask_not_all_caps(col: pd.Series) -> pd.Series:
        return col.apply(lambda s: not is_all_caps(s))

    def mask_num_punct_ratio(col: pd.Series) -> pd.Series:
        return col.apply(lambda s: ratio_numbers_punct(s) <= max_num_punct_ratio)

    rules = [
        ("min_words", mask_min_words),
        ("starts_with_letter", mask_starts_with_letter),
        ("not_all_caps", mask_not_all_caps),
        ("num_punct_ratio_ok", mask_num_punct_ratio),
    ]

    kept_mask = pd.Series(True, index=df.index)
    for rule_name, rule_fn in rules:
        rule_mask_all = pd.Series(True, index=df.index)
        for lang in langs:
            rule_mask_lang = rule_fn(df[lang])
            rule_mask_all &= rule_mask_lang

        removed = int((~rule_mask_all & kept_mask).sum())
        kept_mask &= rule_mask_all
        print(f"Rule '{rule_name}': removed {removed} rows")

    cleaned = df[kept_mask].reset_index(drop=True)
    print(f"Cleaning complete: kept {len(cleaned)}/{before} rows ({len(cleaned)/max(1,before):.1%}).")
    return cleaned

In [ ]:
def save_aligned_outputs(df: pd.DataFrame, langs: Tuple[str, ...], out_dir: str) -> None:
    
    tsv_path = os.path.join(out_dir, f"parallel_{'-'.join(langs)}.tsv")
    df.to_csv(tsv_path, sep="\t", index=False)
    print(f"Saved aligned TSV: {tsv_path} ({len(df)} rows)")

    for lang in langs:
        path = os.path.join(out_dir, f"parallel_{lang}.txt")
        with open(path, "w", encoding="utf-8") as f:
            for line in df[lang].tolist():
                f.write(line.replace("\n", " ").strip() + "\n")
        print(f"Saved {lang} lines: {path} ({len(df)} rows)")


In [ ]:
df = load_aligned_dataframe(DATA_DIR, LANGS)


In [ ]:
df_clean = clean_aligned_dataframe(df, LANGS, MIN_WORDS, MAX_NUM_PUNCT_RATIO)

In [ ]:
save_aligned_outputs(df_clean, LANGS, OUTPUT_DIR)


In [ ]:
print("\nSummary counts:")
for lang in LANGS:
    print(f"{lang}: {len(df_clean)} lines") 